<img src="https://github.com/profitviews/bitcoin-wallet/raw/main/assets/images/demystifying-bitcoin-wallets-digital-interface.webp" style="width:800px"/>

# Demystifying Bitcoin Wallets

> **Read this first: scope and safety**
>
> This notebook walks through the cryptography behind a Bitcoin address: key generation, public key derivation, address encoding, Wallet Import Format (WIF) and signing a transaction. It shows how the pieces work. **It is not a wallet, and it is not a recommendation for how to hold funds.**
>
> To keep the demonstration separate from operational advice, the code:
>
> - **runs on testnet by default.** Testnet coins have no value. Mainnet is an explicit opt-in.
> - **stores the key encrypted** (scrypt + AES-256-GCM) in a file with owner-only permissions, refuses to overwrite an existing key, and keeps the file out of git. That is better than plaintext, but a passphrase-protected file on an internet-connected machine is still a *hot* key.
> - **never prints the private key in full.** Raw key material should not appear in notebook output, logs, screenshots or version control.
> - **builds and signs transactions without broadcasting them** unless you set a flag, and broadcasting on mainnet needs a second, separate flag.
>
> The final section, *What production custody looks like*, sets out the controls this notebook deliberately leaves out.

**TL;DR**: creating a Bitcoin wallet for yourself is very easy - which should not be surprising.  I'll show you how. Operating one *safely* is the hard part, which is why this notebook keeps the two clearly separate. See also [the blog](https://profitview.net/blog/demystifying-bitcoin-wallets).

A year and a half ago, things were on the up.  The COVID crisis was fading and my business had just completed a successful contract.  More excitingly, we were developing a close relationship with our biggest partner: FTX.

What happened then is worth recalling carefully.  If the news in those days had been about a federally licensed and protected US domiciled bank collapsing (like SVB did months later), we would have been _less_ shocked.  If we'd been on our guard _appropriately_ - given the unlicensed and off-shore nature of FTX - the warning signs would have been enough to get our money out well in time.  Even SBF's outrageous "it's a box" interview with Matt Levine should've been enough for us to keep our funds off-exchange.

But we were in good company with our myopia.  Our losses were dwarfed by others who were _themselves_ federally regulated and US domiciled.  Replete as they were with financially licensed risk-managers with top business school educations, they were as flummoxed as we were.  I was ashamed - but double shame on them.  How did it happen?

Call it SBF's reality distortion field. I have a strong personal grudge with him as the result of that.  I'll be watching his sentencing closely.  But it's happened.  The truth is that the mantra of "not your keys, not your coins" has always been true.  I'd previously rolled my eyes at that expression, scoffing at the paranoid crypto maximalists who would spout it.  Not any more.  But what do you - in practice - do about it?

## Out of the Frying Pan?

A whole sub-industry has sprouted around securing your crypto.  Large, successful corporations provide "hardware wallets" in USB sticks which you can put on your key-rings.  You know your crypto is safe - you can feel it in your pocket.  For corporations, there is Copper and Coinbase Custody and Fireblocks.  They will integrate with your crypto back-end and protect you from various crypto specific financial risks.

But isn't crypto supposed to be _less_ risky than traditional finance?  Aren't your funds _safe_ on the blockchain?  Also, who are these companies offering protection from losses?  I was wrong to trust FTX - should I trust _them_?

I wondered about this so I thought I'd find out for myself.  I therefore resolved to 1) create my own Bitcoin wallet 2) automate it in Python.  I'll lay that out below - and, at the end, explain why those custody providers exist, and what they do that a notebook can't.

## You don't even need an _internet connection_

First choose the network.  Everything below works the same way on testnet and mainnet; only a couple of version bytes differ.  Leave it on testnet unless you have a specific reason to change it.

In [1]:
NETWORK = "testnet"  # "testnet" or "mainnet"
assert NETWORK in ("testnet", "mainnet")

KEY_FILE = "private_key.enc"

What you need is a random number.  That, basically, is your Bitcoin wallet.  No really!

It does have to be the *right kind* of random number.  It must come from a cryptographically secure generator: Python's [`secrets`](https://docs.python.org/3/library/secrets.html) module, which reads from the operating system's CSPRNG, and never `random`, which is predictable.  It must also fall in the valid range for the secp256k1 curve, 1 to n-1.  A random 32-byte value is outside that range with probability around 2<sup>-128</sup>, but checking is cheap and correct.

In [2]:
import secrets

SECP256K1_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

def generate_private_key() -> bytes:
    while True:
        candidate = secrets.token_bytes(32)
        if 1 <= int.from_bytes(candidate, "big") < SECP256K1_N:
            return candidate

private_key = generate_private_key()

You'll need to do some arithmetic on this number to make it useful on the Bitcoin network.  But this number *is* the key: it is the only thing anyone needs to spend whatever is sent to the address derived from it.

Which brings the first warning.  That is "your key".  Lose it and the funds are gone for good; leak it and they belong to whoever finds it.  Writing the raw bytes to a plaintext file is one of the most common ways keys leak - through backups, cloud sync, a careless `git add .`, or malware scanning the disk.  So we encrypt it at rest instead:

- **scrypt** turns a passphrase into an encryption key, and is deliberately slow and memory-hungry so that guessing passphrases is expensive.
- **AES-256-GCM** encrypts the key and authenticates it, so a wrong passphrase or a tampered file is detected rather than silently giving you a wrong key.
- The file is created with **owner-only permissions** (`0600`) and the code **refuses to overwrite** an existing key file, since overwriting a funded key is a classic way to lose funds.
- `*.enc` is in `.gitignore`.

In [3]:
import getpass
import json
import os

from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.scrypt import Scrypt

SCRYPT_PARAMS = {"n": 2**17, "r": 8, "p": 1}
MIN_PASSPHRASE_LENGTH = 12

def get_passphrase(confirm: bool = False) -> str:
    # WALLET_PASSPHRASE exists only so the notebook can run unattended (e.g. in CI) on testnet
    passphrase = os.environ.get("WALLET_PASSPHRASE") or getpass.getpass("Key file passphrase: ")
    if len(passphrase) < MIN_PASSPHRASE_LENGTH:
        raise ValueError(f"Use a passphrase of at least {MIN_PASSPHRASE_LENGTH} characters")
    if confirm and "WALLET_PASSPHRASE" not in os.environ:
        if getpass.getpass("Confirm passphrase: ") != passphrase:
            raise ValueError("Passphrases do not match")
    return passphrase

def _derive_key(passphrase: str, salt: bytes, n: int, r: int, p: int) -> bytes:
    return Scrypt(salt=salt, length=32, n=n, r=r, p=p).derive(passphrase.encode())

def save_encrypted_key(private_key: bytes, path: str, passphrase: str) -> None:
    salt, nonce = os.urandom(16), os.urandom(12)
    ciphertext = AESGCM(_derive_key(passphrase, salt, **SCRYPT_PARAMS)).encrypt(nonce, private_key, NETWORK.encode())
    record = {
        "version": 1,
        "network": NETWORK,
        "kdf": "scrypt",
        **SCRYPT_PARAMS,
        "salt": salt.hex(),
        "nonce": nonce.hex(),
        "ciphertext": ciphertext.hex(),
    }
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)  # O_EXCL: never overwrite an existing key
    with os.fdopen(fd, "w") as key_file:
        json.dump(record, key_file)

def load_encrypted_key(path: str, passphrase: str) -> bytes:
    with open(path) as key_file:
        record = json.load(key_file)
    if record["network"] != NETWORK:
        raise ValueError(f"{path} holds a {record['network']} key but NETWORK is {NETWORK}")
    key = _derive_key(passphrase, bytes.fromhex(record["salt"]), record["n"], record["r"], record["p"])
    return AESGCM(key).decrypt(bytes.fromhex(record["nonce"]), bytes.fromhex(record["ciphertext"]), record["network"].encode())

In [4]:
if os.path.exists(KEY_FILE):
    raise FileExistsError(f"{KEY_FILE} already exists. Refusing to overwrite it - move it aside yourself if you are sure it holds no funds.")

save_encrypted_key(private_key, KEY_FILE, get_passphrase(confirm=True))
print(f"Encrypted key written to {KEY_FILE} (mode {oct(os.stat(KEY_FILE).st_mode & 0o777)})")

Encrypted key written to private_key.enc (mode 0o600)


Note that this protects the key *at rest* only.  While the notebook runs, the key sits in plain form in the Python process's memory, and Python offers no reliable way to wipe it.  That is one of several reasons production systems never handle signing keys in a general-purpose interpreter (see the last section).

One thing that's remarkable about all this is that standard Python, run entirely disconnected from the internet, has now given you those "keys" - it's done.

There are Python Bitcoin libraries that can get through the next steps in a couple of lines, but I'll do them using libraries that have no connection with crypto to make the point:

In [5]:
import ecdsa
sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
vk = sk.verifying_key
public_key = b'\x04' + vk.to_string()

In [6]:
import hashlib
import base58

ADDRESS_VERSION = {"mainnet": b'\x00', "testnet": b'\x6f'}

sha256 = hashlib.sha256(public_key).digest()
ripemd160 = hashlib.new('ripemd160', sha256).digest()

network_byte = ADDRESS_VERSION[NETWORK]

network_bitcoin_public_key = network_byte + ripemd160
checksum = hashlib.sha256(hashlib.sha256(network_bitcoin_public_key).digest()).digest()[:4]

bitcoin_address = base58.b58encode(network_bitcoin_public_key + checksum).decode()
print(f"Bitcoin {NETWORK} address: {bitcoin_address}")

Bitcoin testnet address: mrKUHGvJhhfJi1r2j3C3d35dtjq4wuGFuy


Now you have your Bitcoin address.  It can be given to anyone and Bitcoin can be sent to it.  (Testnet addresses of this type start with `m` or `n`; mainnet ones with `1`.)  Unlike the private key, the address is public information.

## How it hangs together

[ecdsa](https://pypi.org/project/ecdsa/) is a pure-Python implementation of [Elliptic Curve Cryptography](https://en.wikipedia.org/wiki/Elliptic-curve_cryptography). ECC was chosen for Bitcoin to create public keys in a way that ensures the private keys can't be reverse engineered.  Note this is what justifies the term "crypto currency": the currency is protected through mathematical cryptography rather than by anti-counterfeiting technology or through the trust of banks.  This _is_ the great innovation underpinning the digital asset industry.

It is used here because its code is easy to follow, not because it is fit for production.  Its own documentation warns that it is not hardened against side-channel attacks such as timing analysis.  Real wallets use [libsecp256k1](https://github.com/bitcoin-core/secp256k1), the constant-time library maintained by Bitcoin Core (which the `bit` library used below wraps).

Having generated a public key from the private key, there's just a little more arithmetic to create a usable Bitcoin address.

[hashlib](https://docs.python.org/3/library/hashlib.html) is part of standard Python.  Suffice it to say that it is a standard way of referring reliably to some data.  In this case that data are your Bitcoin keys.  The steps shown are the ones used for Bitcoin.  They are very standard and reasonable.

The production of the final Bitcoin address uses [base58](https://github.com/keis/base58).  There's nothing remarkable about this.  Its function is "written on the tin". It provides functionality to manipulate numbers in base 58 - "octapentadecimal" if you wish.  We normally use _decimal_ - base 10.  In computer science, _binary_ (base 2), _octal_ (base 8) and _hexadecimal_ (base 16) are often used.  Hexadecimal extends the numerical digits with ABCDEF.  Base 64 is also often used to compress numbers using all letters and numbers (including lower-case and two punctuations) to extend the digit set.  However when those numbers need to be manually copied, there can be visual confusion between some of the letters.  Hence base 58, which excludes e.g. ["0 (zero), O (capital o), I (capital i) and l (lower case L)"](https://digitalbazaar.github.io/base58-spec/)

Note that all the above techniques existed prior to Bitcoin and have no dependence on it.  It can all be executed detached from the internet.

What we've built is a legacy *P2PKH* address from an *uncompressed* public key - the original 2009 format, and the easiest to follow by hand.  Modern wallets use compressed keys, SegWit (`bc1q…`) or Taproot (`bc1p…`) addresses, and hierarchical deterministic (HD) wallets ([BIP32](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki)/[BIP39](https://github.com/bitcoin/bips/blob/master/bip-0039.mediawiki)) that derive every key from a single backed-up seed.

At this point you hold a key that no third party controls - which is what "not your keys, not your coins" is about.  It does **not** follow that you should move meaningful funds to it.  Self-custody swaps *counterparty* risk for *operational* risk: loss, theft, malware, a single point of failure and no recovery process.  Managing that risk is what the custody industry above exists to do, and the last section covers how.

BTW, I'm not using the term "wallet".  As you can see it doesn't really mean anything.  What you have is a _number_ which is referred to as an address.  Since it receives funds, that seems a reasonable name.

### But - so disconnected

What's going on here?  There has been nothing like a "registration" with the Bitcoin network.  How is it that funds can be transferred to some number that I dreamed up, literally, at random?

When you send funds to your new address via the Bitcoin network, the transaction is placed in a block that is then verified by Bitcoin miners (you can look up the exact process).  After that, your address is encoded in the ledger on the blockchain as having those funds added to its balance.

(It is possible that someone _else_ could have, or might later generate, an identical random number - but with a properly generated 256-bit key that is vanishingly unlikely.  The real danger is *weak* randomness: the 2013 Android `SecureRandom` bug and guessable "brain wallets" both led to real funds being stolen.  Hence `secrets` above.)

## Now I want to get my Bitcoin _out_

Anyone can transfer Bitcoin to your address - even if you don't want them to.  That is similar to a bank account.  I suppose it is assumed that no-one objects to more money.  It is when you want to transfer money out - say, in order to pay someone - that you need those "keys".

So let's get back that private key from before.  Decryption needs the passphrase, and fails loudly if it is wrong or the file has been tampered with:

In [7]:
private_key = load_encrypted_key(KEY_FILE, get_passphrase())

There are a few simple steps to take to get it in the right form.  It needs a version prefix to show which network it is for (`0x80` for mainnet, `0xef` for testnet) and a checksum.

In [8]:
WIF_VERSION = {"mainnet": b'\x80', "testnet": b'\xef'}

versioned_private_key = WIF_VERSION[NETWORK] + private_key
checksum = hashlib.sha256(hashlib.sha256(versioned_private_key).digest()).digest()[:4]
versioned_private_key_checksummed = versioned_private_key + checksum

Now we use this to put the private key in [Wallet Import Format (WIF)](https://en.bitcoin.it/wiki/Wallet_import_format).

In [9]:
wif_private_key = base58.b58encode(versioned_private_key_checksummed).decode()

def redact(secret: str, shown: int = 4) -> str:
    return f"{secret[:shown]}…{secret[-shown:]} ({len(secret)} characters)"

print(f"Private key in WIF (redacted): {redact(wif_private_key)}")

Private key in WIF (redacted): 92im…FSyH (51 characters)


Base 58 makes the key easy for a person to copy, which is the point of WIF.  It is also what makes WIF dangerous: anyone who sees the string can spend the funds.  That's why it is redacted above.  Don't print it, log it, screenshot it, paste it into a chat or commit it.

For reference, this is what a full one looks like.  It is the worked example from the Bitcoin wiki's WIF page: `5HueCGU8rMjxEXxiPuD5BDku4MkFqeZyd4dZ1jvhTVqvbTLvyTJ`.  Because it is public, bots watch its address and take anything sent there within seconds - which is exactly what would happen to any key published like this.

Writing a key down on paper (or stamping it into steel) and keeping it offline is a reasonable backup for *small personal* amounts.  Today that would usually be a BIP39 seed phrase rather than a single WIF, and it needs the same physical security as cash.

It's only at this stage that we need to connect to the Bitcoin network and therefore need an internet connection.  `bit` talks to public block explorer APIs, which means those services learn which addresses you are interested in.  That is a privacy leak; serious setups query their own node.

First, a sanity check that the address we derived by hand matches what an established library derives from the same key.  Then check the balance:

In [10]:
from bit import wif_to_key

key = wif_to_key(wif_private_key)
assert key.address == bitcoin_address, "Hand-derived address does not match the library's"

balance = key.get_balance('btc')
print(f"{type(key).__name__} {key.address}: balance = {balance} BTC")

PrivateKeyTestnet mrKUHGvJhhfJi1r2j3C3d35dtjq4wuGFuy: balance = 0 BTC


You can also send funds _from_ the address to another address.  By default the fee comes from https://mempool.space/api/v1/fees/recommended, though you can set a specific fee if you want.

This is where mistakes cost money, because a Bitcoin transaction can't be reversed.  So the cell below is defensive:

- It **builds and signs** the transaction locally but only **broadcasts** it when `BROADCAST = True`.
- Broadcasting on mainnet also needs `ALLOW_MAINNET_BROADCAST = True`, so changing one setting by accident can't send real funds.
- It sends a small amount back to the same address.  If you change the recipient, check it character by character against a trusted source; clipboard-hijacking malware and "address poisoning" both rely on people not doing this.

To try it, get some free coins from a testnet faucet.

In [11]:
from decimal import Decimal
from bit.network import NetworkAPI

recipient = key.address  # Replace with a testnet address you control
amount = Decimal("0.0001")  # BTC
BROADCAST = False
ALLOW_MAINNET_BROADCAST = False

if NETWORK == "mainnet" and BROADCAST and not ALLOW_MAINNET_BROADCAST:
    raise RuntimeError("Refusing to broadcast on mainnet without ALLOW_MAINNET_BROADCAST = True")

if Decimal(balance) < amount:
    print(f"Balance {balance} BTC is below {amount} BTC - fund {key.address} from a {NETWORK} faucet first.")
else:
    signed_tx = key.create_transaction([(recipient, amount, "btc")])
    print(f"Signed transaction built ({len(signed_tx) // 2} bytes).")
    if BROADCAST:
        (NetworkAPI.broadcast_tx_testnet if NETWORK == "testnet" else NetworkAPI.broadcast_tx)(signed_tx)
        print("Broadcast.")
    else:
        print("Not broadcast (BROADCAST = False).")

Balance 0 BTC is below 0.0001 BTC - fund mrKUHGvJhhfJi1r2j3C3d35dtjq4wuGFuy from a testnet faucet first.


This is the basic code for moving funds programmatically, and it can run as part of an algorithmic strategy in [ProfitView](https://profitview.net).  Bear in mind that automating it makes it a *hot wallet*: the key is online and spending is unattended.  Keep only working balances in it, with the bulk held in cold or institutional custody.  Book [a call with us](https://cal.mixmax.com/profitview) and we can walk you through some examples.

There's a bit more work to do the same with, for instance, `USDT` since you'd want to move it on cheaper and faster blockchains than Ethereum, which adds a bit of complexity.  It is very much doable however - but must be the subject of another blog.

## What production custody looks like

Everything above shows how the mechanics work.  None of it is how you should hold funds that matter, whether your own or anyone else's.  This is the gap between the two:

| Concern | This notebook | Production practice |
|---|---|---|
| Key generation | One CSPRNG call in a Python process | Generated inside an HSM, or jointly by MPC parties, in a documented and witnessed key ceremony |
| Key storage | Passphrase-encrypted file on a laptop | The whole key never exists outside secure hardware (HSMs, secure enclaves) or exists only as MPC shares; cold keys are kept air-gapped |
| Single point of failure | One key, one passphrase, one person | Multisig (e.g. 2-of-3) or MPC threshold signing, so no single person or device can move funds |
| Backup and recovery | None beyond the file | Seed phrases or key shares split across locations, with recovery tested regularly |
| Spending controls | A flag in a cell | Policy engine: allow-listed destinations, amount and velocity limits, maker-checker approvals, time delays |
| Signing environment | General-purpose interpreter that can't wipe memory | Hardened, isolated signers; offline signing of [PSBTs](https://github.com/bitcoin/bips/blob/master/bip-0174.mediawiki) for cold storage |
| Fund segregation | Everything in one address | Hot, warm and cold tiers sized to operational need, with client assets kept separate |
| Monitoring | None | Audit logs, on-chain monitoring, reconciliation, incident response, key rotation |
| Assurance | None | Independent audits (SOC 2, ISO 27001), insurance, and regulatory regimes such as NYDFS custody guidance and MiCA |

That gap is why the custodians mentioned at the start exist.  The FTX lesson is not "hold your own keys at any cost".  It is "know exactly who controls the keys, and what controls they operate under".